# Dove vanno i soldi dei comuni italiani?

Analisi della spesa dei comuni italiani dal 2021 al 2025, basata sui dati SIOPE
(Sistema Informativo sulle Operazioni degli Enti Pubblici) della Ragioneria Generale dello Stato.

Dataset: `siope_uscite_comuni` (2021-2025) — pubblico su GCS.

---

In [ ]:
import duckdb
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET threads=4;")

GCS_BASE = "https://storage.googleapis.com/dataciviclab-clean/siope"
ANNI = list(range(2021, 2026))

# Carica tutti gli anni in un'unica vista, filtrando solo COMUNE
parts = []
for a in ANNI:
    url = f"{GCS_BASE}/siope_uscite_comuni/{a}/siope_uscite_comuni_{a}_clean.parquet"
    parts.append(f"SELECT * FROM '{url}'")
union = " UNION ALL ".join(parts)
con.execute(f"CREATE OR REPLACE VIEW tutta_spesa AS {union}")
con.execute("CREATE OR REPLACE VIEW comuni AS SELECT * FROM tutta_spesa WHERE tipo_ente = 'COMUNE'")

tot = con.execute("SELECT count(*) FROM comuni").fetchone()[0]
com_uniq = con.execute("SELECT count(DISTINCT codice_ente) FROM comuni").fetchone()[0]
reg = con.execute("SELECT count(DISTINCT regione) FROM comuni WHERE regione IS NOT NULL AND regione != ''").fetchone()[0]
print(f"Righe: {tot:,} | Comuni: {com_uniq:,} | Regioni: {reg}")

## 1. Trend nazionale della spesa 2021-2025

In [ ]:
trend = con.execute("""
    SELECT anno, ROUND(SUM(importo_eur) / 1e9, 2) AS spesa_mld
    FROM comuni
    WHERE importo_eur > 0
    GROUP BY anno
    ORDER BY anno
""").fetchdf()
print(trend.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(trend['anno'].astype(str), trend['spesa_mld'], color='#2c3e50', width=0.6)
for i, v in enumerate(trend['spesa_mld']):
    ax.text(i, v + 0.3, f'{v:.1f}', ha='center', fontweight='bold', fontsize=11)
ax.set_title('Spesa totale dei comuni italiani — 2021-2025', fontsize=14, fontweight='bold')
ax.set_ylabel('Miliardi di €')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f'))
plt.tight_layout()
plt.savefig('../figures/siope-comuni-spesa_trend_nazionale.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Composizione per macro-categoria

In [ ]:
comp = con.execute("""
    SELECT anno, macro_categoria, ROUND(SUM(importo_eur) / 1e9, 2) AS mld
    FROM comuni
    WHERE macro_categoria IS NOT NULL AND macro_categoria != ''
      AND importo_eur > 0
    GROUP BY anno, macro_categoria
    ORDER BY anno, mld DESC
""").fetchdf()

top_cat = comp[comp['anno'] == 2025].head(8)['macro_categoria'].tolist()
pivot = comp[comp['macro_categoria'].isin(top_cat)].pivot_table(
    index='anno', columns='macro_categoria', values='mld', aggfunc='sum'
)
print(pivot.to_string())

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#2c3e50', '#27ae60', '#e74c3c', '#f39c12', '#3498db', '#9b59b6', '#1abc9c', '#95a5a6']
pivot.plot(kind='bar', stacked=True, ax=ax, color=colors[:len(pivot.columns)], width=0.7)
ax.set_title('Composizione della spesa dei comuni — 2021-2025', fontsize=14, fontweight='bold')
ax.set_ylabel('Miliardi di €')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f'))
ax.legend(title='Macro-categoria', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../figures/siope-comuni-spesa_composizione_categorie.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Spesa per regione (2024, anno completo più recente)

In [ ]:
reg = con.execute("""
    SELECT regione,
           ROUND(SUM(CASE WHEN macro_area = 'Spese correnti' THEN importo_eur ELSE 0 END) / 1e9, 2) AS corrente_mld,
           ROUND(SUM(CASE WHEN macro_area = 'Spese in conto capitale' THEN importo_eur ELSE 0 END) / 1e9, 2) AS capitale_mld,
           ROUND(SUM(importo_eur) / 1e9, 2) AS totale_mld,
           (SUM(CASE WHEN macro_area = 'Personale' THEN importo_eur ELSE 0 END) / SUM(importo_eur) * 100) AS incidenza_personale
    FROM comuni
    WHERE anno = 2024  -- anno completo
      AND regione IS NOT NULL AND regione != ''
      AND importo_eur > 0
    GROUP BY regione
    ORDER BY totale_mld DESC
""").fetchdf()
print(reg[['regione', 'totale_mld', 'corrente_mld', 'capitale_mld']].head(10).to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(reg['regione'], reg['totale_mld'], color='#2c3e50')
ax.set_title('Spesa totale dei comuni per regione — 2024', fontsize=14, fontweight='bold')
ax.set_xlabel('Miliardi di €')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f'))
plt.tight_layout()
plt.savefig('../figures/siope-comuni-spesa_spesa_regioni.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Le voci di spesa più pesanti

In [ ]:
top_voci = con.execute("""
    SELECT descrizione_codice, macro_categoria, ROUND(SUM(importo_eur)/1e9, 2) AS mld
    FROM comuni
    WHERE anno = 2024
      AND descrizione_codice IS NOT NULL AND descrizione_codice != ''
      AND importo_eur > 0
    GROUP BY descrizione_codice, macro_categoria
    ORDER BY mld DESC
    LIMIT 15
""").fetchdf()
print(top_voci[['descrizione_codice', 'macro_categoria', 'mld']].to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 7))
pal = {'Personale': '#2c3e50', 'Acquisto beni e servizi': '#27ae60', 
       'Investimenti fissi': '#e74c3c', 'Trasferimenti correnti': '#3498db',
       'Altre spese': '#95a5a6', 'Imposte e tasse': '#f39c12', 
       'Rimborso prestiti': '#9b59b6', 'Anticipazioni': '#1abc9c',
       'Trasferimenti c/capitale': '#e67e22', 'Contributi investimenti': '#2ecc71'}
colors = [pal.get(c, '#95a5a6') for c in top_voci['macro_categoria']]
ax.barh(range(len(top_voci)), top_voci['mld'], color=colors)
ax.set_yticks(range(len(top_voci)))
ax.set_yticklabels([(t[:55] + '...') if len(t) > 55 else t for t in top_voci['descrizione_codice']], fontsize=8)
ax.set_xlabel('Miliardi di €')
ax.set_title('Top 15 voci di spesa dei comuni — 2024', fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f'))
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in pal.items()]
ax.legend(handles=legend_elements, title='Categoria', loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig('../figures/siope-comuni-spesa_top_voci.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Incidenza del personale sul totale per regione

In [ ]:
personale = con.execute("""
    SELECT regione,
           ROUND(SUM(importo_eur) / 1e6, 1) AS spesa_totale_mln,
           ROUND(SUM(CASE WHEN macro_categoria = 'Personale' THEN importo_eur ELSE 0 END) / 1e6, 1) AS personale_mln,
           ROUND(SUM(CASE WHEN macro_categoria = 'Personale' THEN importo_eur ELSE 0 END) / SUM(importo_eur) * 100, 1) AS incidenza
    FROM comuni
    WHERE anno = 2024
      AND regione IS NOT NULL AND regione != ''
      AND importo_eur > 0
    GROUP BY regione
    HAVING SUM(importo_eur) > 1e8
    ORDER BY incidenza DESC
""").fetchdf()
print(personale[['regione', 'incidenza']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
media = personale['incidenza'].mean()
ax.bar(personale['regione'], personale['incidenza'],
       color=['#e74c3c' if v > media else '#2c3e50' for v in personale['incidenza']])
ax.axhline(y=media, color='#95a5a6', linestyle='--', linewidth=1.5, label=f'Media: {media:.1f}%')
ax.set_title('Incidenza spesa personale sul totale — 2024', fontsize=14, fontweight='bold')
ax.set_ylabel('% sul totale')
ax.legend()
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('../figures/siope-comuni-spesa_incidenza_personale.png', dpi=150, bbox_inches='tight')
plt.show()

con.close()

---
Notebook generato il 2026-06-17. Dati: SIOPE (Ragioneria Generale dello Stato).